# RAG System

## Topic: Dual-Use Dilemma: AI in Offensive vs. Defensive Security

### RAG System Workflow

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           RAG SYSTEM ARCHITECTURE                           │
└─────────────────────────────────────────────────────────────────────────────┘

  ┌──────────────────────┐     ┌──────────────┐      ┌──────────────────────┐
  │   Ingest Documents   │────▶│   Chunker    │────▶│    Hybrid retrieval  │
  │        (PDF)         │     │ (LangChain)  │      │   (Vectors +  BM25)  │
  └──────────────────────┘     └──────────────┘      └──────┬───────────────┘
                                                            │
                                                            │
  ┌──────────────┐     ┌──────────────┐                     │
  │   Ollama     │◀────│  Generator   │◀───────────────────┘
  │ (gemma4:e4b) │     │              │
  └──────────────┘     └──────────────┘
           │
           ▼
  ┌────────────────┐       ┌────────────────────┐
  │     Answer     │─────▶│     Evaluation     │
  │                │       │  (RAGAS + Ollama)  │
  └────────────────┘       └────────────────────┘
```

## Research Questions

### RQ1: 

In [1]:
# Setup and Imports
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('.').absolute()))

from src.rag_system import RAGSystem
print('RAG System imported successfully')

W0524 12:49:31.935000 58280 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



RAG System imported successfully


## Initialize RAG System and preload model on ollama

In [2]:
# Initialize RAG system with source directory
SOURCE_DIR = Path('sources')

rag = RAGSystem(source_dir=SOURCE_DIR)
print('RAG System initialized')

# Check Ollama and preload model
from src.ollama_manager import OllamaManager

print("\n=== Ollama Status Check ===")
ollama_mgr = OllamaManager(
    base_url=rag.generator.config.base_url,
    model=rag.generator.config.model
)

status = ollama_mgr.get_status_summary()
if status["ollama_available"]:
    print("✅ Ollama is running")
    print(f"Available models: {status['available_models']}")

    if status["matching_model"]:
        print(f"✅ Model '{status['matching_model']}' is available")
        print(f"\n⏳ Loading model into memory...")
        success, elapsed = ollama_mgr.preload_model(status["matching_model"])
        if success:
            print(f"✅ Model ready ({elapsed:.1f}s)")
            rag.generator.config.model = status["matching_model"]
        else:
            print(f"⚠️ Model may not be fully loaded")
    else:
        print(f"⚠️ Configured model '{status['configured_model']}' not found")
else:
    print("❌ Ollama is NOT running")
    print("   Start with: `ollama serve`")

RAG System initialized

=== Ollama Status Check ===
✅ Ollama is running
Available models: ['nomic-embed-text:latest', 'gemma4:e4b', 'qwen3.5:4b', 'mistrallite:7b', 'mistral-openorca:7b', 'mistral:7b', 'qwen2:7b', 'qwen3:8b', 'qwen2.5:7b', 'llama3:8b', 'llama3.2:3b', 'gemma3:4b', 'gemma:7b', 'gemma2:9b', 'gemini-3-flash-preview:cloud', 'gemini-3-flash-preview:latest', 'llama3.1:8b', 'mistral-nemo:12b']
✅ Model 'gemma4:e4b' is available

⏳ Loading model into memory...
✅ Model ready (151.2s)


### 1. Ingest Documents

- **Load**: PDF files from `sources/` directory → extracts text per page
- **Chunk**: Use `RecursiveCharacterTextSplitter` from `langchain_text_splitters` python library to splits text into ~1500 char chunks with 200 char overlap
- **Index**: 
  - **Vector Store**: `BAAI/bge-large-en-v1.5` embeddings model → FAISS index for semantic search
  - **BM25**: `rank_bm25` python library for keyword matching

Using framework like LangChain ([LangChain AI, 2022](#ref3)) is extremely effective to ingest large cybersecurity resources and process them into smaller, semantically meaningful text segments. Selecting 1,500 characters as maxiumn chunk size provides a balanced approach which small enough to avoid embedding irrelevant information while maintain semantic accuracy without breaking up dense concepts ([Zhao et al., 2025](#ref1); [Blefari et al., 2025](#ref2)).

Dense retrieval uses deep learning models to convert sentences into lists of numbers (vectors). This helps the system understand the actual meaning behind a question instead of just matching words. By using Facebook AI Similarity Search ([FAISS](#ref8)), the system can search through millions of these text pieces very quickly. However, this method can sometimes struggle to find exact words, technical codes, or specific error logs ([Asai et al., 2024](#ref4)).

To fix what dense retrieval misses, using BM25 ([Robertson & Zaragoza, 2009](#ref7)) to catch exact matching words. BM25 is an improved version of basic word counting (TF-IDF). It handles different document lengths and caps scores so repeating a word too many times does not mess up the results. Using the lightweight Python Library `rank_bm25` ([rank_bm25](#ref9)) helps the system catch literal word overlaps, error names, and short technical terms ([Hsu et al., 2025](#ref5); [Karpukhin et al., 2020](#ref6)).

Research proves that using only one search tool can cause the system to miss good documents or pull up wrong ones. As shown by the creators of Dense Passage Retrieval (DPR) ([Karpukhin et al., 2020](#ref6)), mixing exact word scores with deep meaning maps gives much better results than using just one method. Gathering results from both types of search creates a cleaner, highly accurate group of documents that helps the language model give much more reliable answers ([Hsu et al., 2025](#ref5); [Karpukhin et al., 2020](#ref6)).

### Key Components
| Component | File | Purpose |
|-----------|------|---------|
| `chunker.py` | `create_chunks()` | LangChain text splitting |
| `vector_store.py` | `VectorStore` | FAISS + `BAAI/bge-large-en-v1.5` embeddings |
| `bm25_retriever.py` | `BM25Retriever` | BM25 keyword search |

In [3]:
# Ingest and index all documents
stats = rag.ingest_documents(SOURCE_DIR)
print(f"Ingested {stats['documents_loaded']} documents")
print(f"Created {stats['chunks_created']} chunks")
print(f"System stats: {rag.get_stats()}")

Loading cached index for 11 files...
Loading vector index from .rag_index\vectors.faiss...
Loading BM25 index from .rag_index\bm25.json...
Loaded 948 chunks from .rag_index\chunks.json
Ingested 11 documents
Created 946 chunks
System stats: {'indexed': True, 'document_count': 948, 'model': 'gemma4:e4b', 'embedding_model': 'BAAI/bge-large-en-v1.5', 'index_dir': '.rag_index'}


## Evaluation

### 2. Query Pipeline
1. **Embed Query**: Convert user question to embedding vector
2. **Hybrid Retrieval**: 
   - Semantic: Top-50 results from FAISS vector similarity
   - Keyword: Top-50 results from BM25 scoring

3. **Reranking**
   - **RRF Fusion** ([Cormack et al., 2009](#ref10)): Reciprocal Rank Fusion (k=60) combines both ranked lists and get the Top-3 chunks

   $$RRF = \sum \frac{1}{k + rank}$$

   - **Cross-Encoder**: Put the user question and document document chunks from sematic and keyword search into AI at the same time. The AI performs intense, token-by-token attention analysis
   - **Hybrid**: Take Top-10 chunks from RRF then in `Cross-Encoder` use only those chunks with user question
   
3. **Generate**: question + Retrieved chunks → Ollama (gemma4:e4b) → Answer

In this RAG system, I implement a two-stage hybrid retrieval pipeline that balances high recall with deep semantic precision ([Caraman et al., 2026](#ref12)). First, I use Reciprocal Rank Fusion (RRF) ([Cormack et al., 2009](#ref10)) to combine the results from dense vector search and keyword search (BM25). Because these two search methods grade text in completely different ways, we can't simply add their raw scores together. Instead, as [Bruch et al., 2023](#ref11) explain, RRF ignores the complicated raw scores entirely and only looks at a document's position within each list, effectively catching a broad, diverse set of candidates. However, because RRF is blind to the actual content of the documents, I feed its top-ranked candidates into a [Cross-Encoder](#ref13) re-ranking model. The Cross-Encoder performs full cross-attention between the query and text, acting as a high-precision filter that catches deep context and subtle nuances that rank-based aggregation misses before the final context window is generated.

### 3. RAGAS Evaluation Metrics
| Metric | Description |
|--------|-------------|
| **Faithfulness** | Does the answer contain only facts from the context? |
| **Answer Relevancy** | How relevant is the answer to the question? |
| **Context Relevance** | Does contexts address the qurey? |

### Key Components
| Component | File | Purpose |
|-----------|------|---------|
| `hybrid_retriever.py` | `HybridRetriever` | RRF fusion of both |
| `generator.py` | `OllamaGenerator` | Ollama API client |
| `evaluator.py` | `RAGASEvaluator` | RAGAS metrics computation |

In [ ]:
# RAGAS Evaluation
from src.test_case import CASES
from src.evaluator import RAGASEvaluator

# Initialize evaluator with RAG system
# rerank_mode include Reciprocal Rank Fusion (rrf), Neural Reranking (neural) and RRF + Neural Reranking (hybrid)
evaluator = RAGASEvaluator(rag, rerank_mode = "rrf")
print("Evaluator initialized")

c:\Users\sam99\AppData\Local\Programs\Python\Python311\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


Evaluator initialized


In [5]:
# Configure evaluation settings
from src.config import config

print("Evaluation configuration:")
print(f"  Model: {config.eval_llm.model}")
print(f"  Base URL: {config.eval_llm.base_url}")
print(f"  Metrics: {evaluator.metrics}")

Evaluation configuration:
  Model: gemma4:e4b
  Base URL: http://localhost:11434/v1
  Metrics: ['faithfulness', 'answer_relevancy', 'context_relevance']


In [6]:
# Run evaluation on all test cases
results = evaluator.run_batch(CASES)
print(f"Evaluated {len(results)} test cases")

[Evaluator] Starting batch evaluation of 10 cases...
[Evaluator] [1/10] Processing case...
[Evaluator] [1/10] Case complete
[Evaluator] [2/10] Processing case...
[Evaluator] [2/10] Case complete
[Evaluator] [3/10] Processing case...
[Evaluator] [3/10] Case complete
[Evaluator] [4/10] Processing case...
[Evaluator] [4/10] Case complete
[Evaluator] [5/10] Processing case...
[Evaluator] [5/10] Case complete
[Evaluator] [6/10] Processing case...
[Evaluator] [6/10] Case complete
[Evaluator] [7/10] Processing case...
[Evaluator] [7/10] Case complete
[Evaluator] [8/10] Processing case...
[Evaluator] [8/10] Case complete
[Evaluator] [9/10] Processing case...
[Evaluator] [9/10] Case complete
[Evaluator] [10/10] Processing case...
[Evaluator] [10/10] Case complete
[Evaluator] Batch evaluation complete: 10 results
Evaluated 10 test cases


In [7]:
# Display evaluation results
evaluator.print_results(results)
evaluator.save_results(results)


RAGAS EVALUATION RESULTS
Question                                           Faith    Relev    CtxRel  
--------------------------------------------------------------------------------
How does prompt injection manipulate the origin... 1.00    0.78    1.00
  [Answer]: Prompt injection manipulates system prompts by altering the input to cause unauthorized actions or s...
  [1] Securing-Agentic-Applications-Guide-1.0.pdf, Page 30 (score: 0.032)
  [2] Securing-Agentic-Applications-Guide-1.0.pdf, Page 66 (score: 0.031)
  [3] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 10 (score: 0.030)
What specific security risks are associated wit... 1.00    0.88    1.00
  [Answer]: Persistent memory in agentic AI systems increases the risk of several issues, including context pois...
  [1] Securing-Agentic-Applications-Guide-1.0.pdf, Page 31 (score: 0.032)
  [2] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 27 (score: 0.027)
  [3] Artificial Intelligence Risk Management Fra

## References

<a id="ref1">[1]</a>  Zhao, C., De Maria, R., Kumarage, T., Chaudhary, K. S., Agrawal, G., Li, Y., Park, J., Deng, Y., Chen, Y. C., & Liu, H. (2025). CyberBOT: Towards reliable cybersecurity education via ontology-grounded retrieval augmented generation. arXiv preprint arXiv:2504.00389. https://doi.org/10.48550/arxiv.2504.00389

<a id="ref2">[2]</a> Blefari, F., Cosentino, C., Pironti, F. A., Furfaro, A., & Marozzo, F. (2025). CyberRAG: An Agentic RAG cyber attack classification and reporting tool. arXiv. https://doi.org/10.1016/j.future.2025.108186

<a id="ref3">[3]</a> LangChain AI. (2022). LangChain (Version 0.3.7) [Computer software]. GitHub. https://github.com/langchain-ai/langchain

<a id="ref4">[4]</a> Asai, A., Wu, Z., Wang, Y., Sil, A., & Hajishirzi, H. (2024). Self-RAG: Learning to retrieve, generate, and critique through self-reflection. In Proceedings of the International Conference on Learning Representations (ICLR). https://openreview.net/forum?id=hSyW5go0v8

<a id="ref5">[5]</a> Hsu, H.-L., & Tzeng, J. (2025). DAT: Dynamic Alpha Tuning for Hybrid Retrieval in Retrieval-Augmented Generation. arXiv preprint arXiv:2503.23013. https://doi.org/10.48550/arxiv.2503.23013

<a id="ref6">[6]</a> Karpukhin, V., Oğuz, B., Min, S., Lewis, P., Wu, L., Edunov, S., Chen, D., & Yih, W. (2020). Dense Passage Retrieval for Open-Domain Question Answering. In Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (EMNLP) (pp. 6769–6781). Association for Computational Linguistics.

<a id="ref7">[7]</a> Robertson, S., & Zaragoza, H. (2009). The Probabilistic Relevance Framework: BM25 and Beyond. Foundations and Trends® in Information Retrieval, 3(4), 333–389. https://doi.org/10.1561/1500000019

<a id="ref8">[8]</a> Facebook AI Research. (n.d.). Faiss: A library for efficient similarity search and clustering of dense vectors [Computer software]. GitHub. https://github.com/facebookresearch/faiss

<a id="ref9">[9]</a> Trotman, A. (2019). rank-bm25: A collection of BM25 algorithms in Python [Computer software]. GitHub. https://github.com/dorianbrown/rank_bm25

<a id="ref10">[10]</a> Cormack, G. V., Clarke, C. L. A., & Buettcher, S. (2009). Reciprocal rank fusion outperforms condorcet and individual rank learning methods. In Proceedings of the 32nd International ACM SIGIR Conference on Research and Development in Information Retrieval (pp. 758–759). Association for Computing Machinery. https://doi.org/10.1145/1571941.1572114

<a id="ref11">[11]</a> Bruch, S., Gai, S., & Ingber, A. (2023). An Analysis of Fusion Functions for Hybrid Retrieval. ACM Transactions on Information Systems, 42, 1–35. https://doi.org/10.1145/3596512

<a id="ref12">[12]</a> Caraman, D. M. (2026). Caraman at SemEval-2026 Task 8: Three-Stage Multi-Turn Retrieval with Query Rewriting, Hybrid Search, and Cross-Encoder Reranking. arXiv. https://arxiv.org/pdf/2605.12028

<a id="ref13">[13]</a> Sentence Transformers. (n.d.). Cross-encoder. Hugging Face. Retrieved May 22, 2026, from https://huggingface.co/cross-encoder
